# Snow DA Impact: `DAv8` vs `OLv8` (M21C Land Sweeper)

This notebook compares two monthly land-sweeper summary files and focuses on the **impact of snow DA**:

- snow state changes: `SNOMASLAND`, `FRLANDSNO`
- near-surface thermal response: `TSOIL1`
- soil moisture response: `SFMC`, `RZMC`

The notebook is intentionally simple and heavily commented so it is easy to modify.


## What This Notebook Produces

1. Basic file/grid compatibility checks (OL vs DA)
2. Regional monthly time series (OL, DA, and `DA-OL`)
3. Monthly climatology of `DA-OL`
4. Seasonal lon/lat scatter maps of `DA-OL` (DJF/MAM/JJA/SON)
5. Snow-conditioned and frozen-conditioned distributions of `DA-OL`
6. Spring snow impact vs summer root-zone soil moisture impact (carryover)
7. Optional period summaries using custom time windows

Notes:
- These files are monthly means, so diagnostics here are **state differences / climatological impacts**, not event-scale increments.
- Means are **unweighted tile means** unless you add tile area weights.


In [ ]:
# -------------------------
# Imports + user configuration
# -------------------------
# Set thread limits before importing numpy/xarray. This can help on systems
# where OpenMP/BLAS thread startup causes issues in notebooks or restricted envs.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

try:
    import dask  # noqa: F401
    HAVE_DASK = True
except Exception:
    HAVE_DASK = False

# -------------------------
# Input files
# -------------------------
DATA_DIR = Path("/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2")
DA_FILE = DATA_DIR / "DAv8_land_variables_2000_2024_compressed.nc"
OL_FILE = DATA_DIR / "OLv8_land_variables_2000_2024_compressed.nc"

# -------------------------
# Output controls
# -------------------------
OUTPUT_DIR = REPO_ROOT / "projects/M21C_ls/output/snow_da_impact"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGURES = True
FIG_DPI = 180

# -------------------------
# Variables to analyze
# -------------------------
# Keep this list small for a fast first pass. Add more later if needed.
ANALYSIS_VARS = [
    "SNOMASLAND",
    "FRLANDSNO",
    "TSOIL1",
    "SFMC",
    "RZMC",
]
CONTEXT_VARS = ["PRECTOTCORRLAND"]  # optional, used if you want extra context plots later

# -------------------------
# Thresholds / masks
# -------------------------
# User requested snow/no-snow threshold = 0.5 on fractional snow cover.
SNOW_FRAC_THRESHOLD = 0.5

# Frozen threshold is configurable. 273.15 K is a strict freezing point.
# If you want a looser "cold soil" threshold, change this (e.g., 277.15 K).
FROZEN_TSOIL1_THRESHOLD_K = 273.15

# -------------------------
# Regions for simple summary means
# -------------------------
# These are unweighted tile masks based on latitude only.
REGION_DEFS = {
    "Global": (-90.0, 90.0),
    "NH": (0.0, 90.0),
    "SH": (-90.0, 0.0),
    "NH (30-75N)": (30.0, 75.0),
    "High NH (45-75N)": (45.0, 75.0),
}

# -------------------------
# Custom period windows (inclusive start/end) for optional summaries
# -------------------------
PERIOD_WINDOWS = [
    ("2000-06 to 2007-05", "2000-06-01", "2007-05-31"),
    ("2007-06 to 2015-03", "2007-06-01", "2015-03-31"),
    ("2015-04 to 2024-05", "2015-04-01", "2024-05-31"),
]

# -------------------------
# Performance controls
# -------------------------
USE_DASK_CHUNKS = HAVE_DASK
TIME_CHUNK = 12        # monthly data -> 1 year chunks are a good default
TILE_CHUNK = 20000     # moderate tile chunk for memory balance

# Map scatter plotting can be slow with all tiles. Increase stride to thin points.
MAP_TILE_STRIDE = 1

# Random sample limits for distribution and scatter plots (to keep plots responsive).
RANDOM_SEED = 42
BOXPLOT_SAMPLE_MAX = 150_000
SCATTER_SAMPLE_MAX = 250_000

print(f"HAVE_DASK={HAVE_DASK}")
print(f"USE_DASK_CHUNKS={USE_DASK_CHUNKS}")
print(f"DA_FILE={DA_FILE}")
print(f"OL_FILE={OL_FILE}")
print(f"OUTPUT_DIR={OUTPUT_DIR.resolve()}")
print(f"SNOW_FRAC_THRESHOLD={SNOW_FRAC_THRESHOLD}")
print(f"FROZEN_TSOIL1_THRESHOLD_K={FROZEN_TSOIL1_THRESHOLD_K}")


## Helper Functions

These helpers keep the analysis cells readable and centralize assumptions (compatibility checks, masking, plotting, and sampling).


In [ ]:
# -------------------------
# Helper functions
# -------------------------
def open_land_sweeper_dataset(path, var_names, use_dask_chunks=True, time_chunk=12, tile_chunk=20000):
    """Open one compressed monthly land-sweeper NetCDF file and keep only needed variables."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing file: {path}")

    chunks = None
    if use_dask_chunks:
        chunks = {"time": time_chunk, "tile": tile_chunk}

    try:
        ds = xr.open_dataset(path, decode_times=True, chunks=chunks)
    except TypeError:
        # Some xarray versions complain if chunks=None is passed in a certain way.
        ds = xr.open_dataset(path, decode_times=True)
    except Exception as exc:
        if use_dask_chunks:
            warnings.warn(
                f"Chunked open failed for {path} ({exc}). Falling back to unchunked open; "
                "this may be slower and use more memory."
            )
            ds = xr.open_dataset(path, decode_times=True)
        else:
            raise

    keep = [v for v in var_names if v in ds.data_vars]
    missing = sorted(set(var_names) - set(keep))
    if missing:
        warnings.warn(f"{Path(path).name}: missing requested variables: {missing}")

    # Keep coordinates needed for plotting and region masks.
    return ds[keep + ["lat", "lon"]]


def assert_compatible_ol_da(ol_ds, da_ds, vars_expected=None):
    """Validate that OL and DA files have matching dimensions and coordinates."""
    if ol_ds.sizes.get("time") != da_ds.sizes.get("time"):
        raise ValueError(f"Time dimension mismatch: OL={ol_ds.sizes.get('time')} DA={da_ds.sizes.get('time')}")
    if ol_ds.sizes.get("tile") != da_ds.sizes.get("tile"):
        raise ValueError(f"Tile dimension mismatch: OL={ol_ds.sizes.get('tile')} DA={da_ds.sizes.get('tile')}")

    # Exact time coordinate match is required for a clean DA-OL difference.
    if not np.array_equal(pd.to_datetime(ol_ds["time"].values), pd.to_datetime(da_ds["time"].values)):
        raise ValueError("Time coordinates differ between OL and DA")

    # Lat/lon should also match tile-for-tile.
    if not np.allclose(np.asarray(ol_ds["lat"].values), np.asarray(da_ds["lat"].values), equal_nan=True):
        raise ValueError("Tile latitude coordinates differ between OL and DA")
    if not np.allclose(np.asarray(ol_ds["lon"].values), np.asarray(da_ds["lon"].values), equal_nan=True):
        raise ValueError("Tile longitude coordinates differ between OL and DA")

    if vars_expected is not None:
        missing_ol = [v for v in vars_expected if v not in ol_ds.data_vars]
        missing_da = [v for v in vars_expected if v not in da_ds.data_vars]
        if missing_ol or missing_da:
            raise ValueError(f"Missing vars -> OL: {missing_ol}, DA: {missing_da}")


def maybe_compute(obj):
    """Compute dask-backed xarray objects; no-op for eager arrays."""
    return obj.compute() if hasattr(obj, "compute") else obj


def region_mask_from_lat(lat_da, lat_min, lat_max):
    """Simple latitude-only tile mask."""
    return (lat_da >= lat_min) & (lat_da <= lat_max)


def tile_mean_timeseries(ds, var_name, tile_mask=None):
    """Monthly mean over tiles for one variable (unweighted)."""
    x = ds[var_name]
    if tile_mask is not None:
        x = x.where(tile_mask)
    return x.mean(dim="tile", skipna=True)


def monthly_clim(da_time_series):
    """Return 12-month climatology from a 1D monthly series."""
    return da_time_series.groupby("time.month").mean(dim="time", skipna=True)


def seasonal_mean_by_tile(da_tile_time):
    """Return seasonal mean by tile (dims: season, tile)."""
    out = da_tile_time.groupby("time.season").mean(dim="time", skipna=True)
    order = [s for s in ["DJF", "MAM", "JJA", "SON"] if s in out["season"].values]
    return out.sel(season=order)


def robust_symmetric_limits(values, pct=99.0):
    """Symmetric color limits around zero for difference maps."""
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return -1.0, 1.0
    vmax = np.nanpercentile(np.abs(arr), pct)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = np.nanmax(np.abs(arr)) if arr.size else 1.0
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    return -float(vmax), float(vmax)


def savefig_if_enabled(fig, filename, output_dir=OUTPUT_DIR, save=True, dpi=180):
    """Save a figure and print the path so reruns are easy to track."""
    if not save:
        return None
    p = Path(output_dir) / filename
    fig.savefig(p, dpi=dpi, bbox_inches="tight")
    print(f"Wrote figure: {p}")
    return p


def plot_lonlat_scatter_map_panels(seasonal_tile_da, lat, lon, title, units="", cmap="RdBu_r", stride=1, s=4):
    """Plot 4 seasonal lon/lat scatter panels without requiring cartopy."""
    seasons = [str(s0) for s0 in seasonal_tile_da["season"].values]

    lat_np = np.asarray(lat.values, dtype=float)
    lon_np = np.asarray(lon.values, dtype=float)

    # Optional point thinning for speed.
    if stride is None or stride < 1:
        stride = 1
    sl = slice(None, None, int(stride))
    lat_plot = lat_np[sl]
    lon_plot = lon_np[sl]

    vals_all = np.asarray(seasonal_tile_da.values, dtype=float)[:, sl]
    vmin, vmax = robust_symmetric_limits(vals_all, pct=99.0)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    fig, axes = plt.subplots(2, 2, figsize=(14, 7), constrained_layout=True)
    axes = axes.ravel()

    sc = None
    for ax, season in zip(axes, seasons):
        vals = np.asarray(seasonal_tile_da.sel(season=season).values, dtype=float)[sl]
        m = np.isfinite(vals) & np.isfinite(lat_plot) & np.isfinite(lon_plot)
        sc = ax.scatter(lon_plot[m], lat_plot[m], c=vals[m], s=s, cmap=cmap, norm=norm, linewidths=0)
        ax.set_title(season)
        ax.set_xlim(-180, 180)
        ax.set_ylim(-90, 90)
        ax.set_xlabel("Lon")
        ax.set_ylabel("Lat")
        ax.grid(alpha=0.2)

    # If fewer than 4 seasons are present, hide extra axes.
    for j in range(len(seasons), len(axes)):
        axes[j].axis("off")

    if sc is not None:
        cbar = fig.colorbar(sc, ax=axes.tolist(), shrink=0.9)
        cbar.set_label(units if units else "DA - OL")
    fig.suptitle(title, fontsize=14)
    return fig


def sample_finite(values, max_n=100_000, seed=42):
    """Randomly subsample a 1D array of finite values for plotting."""
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size <= max_n:
        return arr
    rng = np.random.default_rng(seed)
    idx = rng.choice(arr.size, size=max_n, replace=False)
    return arr[idx]


def sample_finite_pairs(x, y, max_n=200_000, seed=42):
    """Randomly subsample finite x/y pairs for scatter or hexbin diagnostics."""
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if x.size <= max_n:
        return x, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(x.size, size=max_n, replace=False)
    return x[idx], y[idx]


def build_window_bounds(windows):
    """Convert (name, start, end) strings to Timestamp bounds (inclusive)."""
    out = []
    for name, start, end in windows:
        out.append((name, pd.Timestamp(start), pd.Timestamp(end)))
    return out


def seasonal_year_mean_for_months(da_time_tile, months):
    """Mean over selected months for each year; output dims: year, tile.

    This helper assumes the months are within a single calendar year (e.g., MAM or JJA).
    It is intentionally simple and avoids DJF year-boundary logic.
    """
    sel = da_time_tile.where(da_time_tile["time"].dt.month.isin(months), drop=True)
    return sel.groupby("time.year").mean(dim="time", skipna=True)


## Open OL and DA Files, Validate Compatibility, Build `DA-OL`

This is the core setup cell. It opens the two files lazily (chunked if `dask` is available), checks that the coordinates match, and creates a difference dataset used throughout the notebook.


In [ ]:
# -------------------------
# Load datasets and validate compatibility
# -------------------------
request_vars = ANALYSIS_VARS + [v for v in CONTEXT_VARS if v not in ANALYSIS_VARS]

ol = open_land_sweeper_dataset(
    OL_FILE,
    request_vars,
    use_dask_chunks=USE_DASK_CHUNKS,
    time_chunk=TIME_CHUNK,
    tile_chunk=TILE_CHUNK,
)
da = open_land_sweeper_dataset(
    DA_FILE,
    request_vars,
    use_dask_chunks=USE_DASK_CHUNKS,
    time_chunk=TIME_CHUNK,
    tile_chunk=TILE_CHUNK,
)

# Keep only variables available in both files.
common_vars = [v for v in ANALYSIS_VARS if (v in ol.data_vars and v in da.data_vars)]
if len(common_vars) == 0:
    raise RuntimeError("No common analysis variables found in both OL and DA files")

assert_compatible_ol_da(ol, da, vars_expected=common_vars)

# Difference dataset used for impact diagnostics.
# Positive means DA > OL.
d = da[common_vars] - ol[common_vars]
d = d.assign_coords(time=ol["time"], tile=ol["tile"], lat=ol["lat"], lon=ol["lon"])

time_vals = pd.to_datetime(ol["time"].values)
print(f"Common variables: {common_vars}")
print(f"time: {time_vals[0].date()} to {time_vals[-1].date()} ({len(time_vals)} monthly steps)")
print(f"tile count: {ol.sizes['tile']}")
print(f"lat range: {float(ol['lat'].min()):.3f} to {float(ol['lat'].max()):.3f}")
print(f"lon range: {float(ol['lon'].min()):.3f} to {float(ol['lon'].max()):.3f}")
print("OL and DA files are compatible for DA-OL analysis.")


## Quick Sanity Checks (Variable Ranges)

This cell computes simple min/max/mean diagnostics for OL, DA, and `DA-OL` to catch obvious unit/sign mistakes before plotting.


In [ ]:
# -------------------------
# Quick numeric sanity checks
# -------------------------
sanity_rows = []
for tag, ds in [("OL", ol), ("DA", da), ("DA-OL", d)]:
    for var in common_vars:
        x = ds[var]
        # Compute small reductions only (cheap compared to loading the full array into memory).
        x_min = float(maybe_compute(x.min(skipna=True)).values)
        x_max = float(maybe_compute(x.max(skipna=True)).values)
        x_mean = float(maybe_compute(x.mean(dim=("time", "tile"), skipna=True)).values)
        sanity_rows.append({
            "dataset": tag,
            "variable": var,
            "min": x_min,
            "mean": x_mean,
            "max": x_max,
            "units": str(ds[var].attrs.get("units", "")),
        })

sanity_df = pd.DataFrame(sanity_rows)
display(sanity_df)


## Build Region Masks and Regional Monthly Time Series

These are simple unweighted tile means by latitude bands. They give a fast first look at where snow DA changes the land state over time.


In [ ]:
# -------------------------
# Region masks (latitude-only)
# -------------------------
lat = ol["lat"]
lon = ol["lon"]

region_masks = {}
for name, (lat_min, lat_max) in REGION_DEFS.items():
    region_masks[name] = region_mask_from_lat(lat, lat_min, lat_max)

print("Region masks:")
for name, mask in region_masks.items():
    n_tiles = int(maybe_compute(mask.sum()).values)
    print(f"  {name:>22s}: {n_tiles:7d} tiles")

# -------------------------
# Regional time series (monthly means across tiles)
# -------------------------
# We store these in a tidy dataframe so plotting is straightforward.
ts_rows = []
region_names = list(region_masks.keys())

for region_name in region_names:
    mask = region_masks[region_name]
    for var in common_vars:
        ol_ts = maybe_compute(tile_mean_timeseries(ol, var, tile_mask=mask))
        da_ts = maybe_compute(tile_mean_timeseries(da, var, tile_mask=mask))
        d_ts = maybe_compute(tile_mean_timeseries(d, var, tile_mask=mask))

        t = pd.to_datetime(ol_ts["time"].values)
        for i, tt in enumerate(t):
            ts_rows.append({
                "time": tt,
                "region": region_name,
                "variable": var,
                "OL": float(ol_ts.values[i]) if np.isfinite(ol_ts.values[i]) else np.nan,
                "DA": float(da_ts.values[i]) if np.isfinite(da_ts.values[i]) else np.nan,
                "DA_minus_OL": float(d_ts.values[i]) if np.isfinite(d_ts.values[i]) else np.nan,
            })

region_ts_df = pd.DataFrame(ts_rows)
print(f"region_ts_df rows: {len(region_ts_df)}")
display(region_ts_df.head(12))


## Regional Time-Series Plots (OL, DA, and `DA-OL`)

Default focus region: `NH (30-75N)` because snow-DA impacts are usually clearest in this latitude band.


In [ ]:
# -------------------------
# Plot regional time series for a snow-focused region
# -------------------------
FOCUS_REGION = "NH (30-75N)"
if FOCUS_REGION not in set(region_ts_df["region"]):
    raise KeyError(f"Region not found: {FOCUS_REGION}")

# File-safe region tag for figure names.
focus_region_tag = "".join(ch if ch.isalnum() else "_" for ch in FOCUS_REGION).strip("_")

# Figure A: snow-state variables
vars_snow = [v for v in ["SNOMASLAND", "FRLANDSNO"] if v in common_vars]
if vars_snow:
    fig, axes = plt.subplots(len(vars_snow), 2, figsize=(16, 4 * len(vars_snow)), sharex=True, constrained_layout=True)
    axes = np.atleast_2d(axes)

    for r, var in enumerate(vars_snow):
        sub = region_ts_df[(region_ts_df["region"] == FOCUS_REGION) & (region_ts_df["variable"] == var)].sort_values("time")
        t = pd.to_datetime(sub["time"])

        ax = axes[r, 0]
        ax.plot(t, sub["OL"], label="OL", lw=1.4)
        ax.plot(t, sub["DA"], label="DA", lw=1.4)
        ax.set_title(f"{FOCUS_REGION}: {var} (OL and DA)")
        ax.set_ylabel(var)
        ax.grid(alpha=0.25)
        if r == 0:
            ax.legend(loc="best")

        ax = axes[r, 1]
        ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)
        ax.plot(t, sub["DA_minus_OL"], color="#C44E52", lw=1.2)
        ax.set_title(f"{FOCUS_REGION}: {var} (DA - OL)")
        ax.set_ylabel("DA - OL")
        ax.grid(alpha=0.25)

    for ax in axes[-1, :]:
        ax.set_xlabel("Time")

    savefig_if_enabled(fig, f"snow_da_timeseries_{focus_region_tag}_snowvars.png", dpi=FIG_DPI, save=SAVE_FIGURES)
    plt.show()

# Figure B: land-response variables
vars_land = [v for v in ["TSOIL1", "SFMC", "RZMC"] if v in common_vars]
if vars_land:
    fig, axes = plt.subplots(len(vars_land), 2, figsize=(16, 4 * len(vars_land)), sharex=True, constrained_layout=True)
    axes = np.atleast_2d(axes)

    for r, var in enumerate(vars_land):
        sub = region_ts_df[(region_ts_df["region"] == FOCUS_REGION) & (region_ts_df["variable"] == var)].sort_values("time")
        t = pd.to_datetime(sub["time"])

        ax = axes[r, 0]
        ax.plot(t, sub["OL"], label="OL", lw=1.2)
        ax.plot(t, sub["DA"], label="DA", lw=1.2)
        ax.set_title(f"{FOCUS_REGION}: {var} (OL and DA)")
        ax.set_ylabel(var)
        ax.grid(alpha=0.25)
        if r == 0:
            ax.legend(loc="best")

        ax = axes[r, 1]
        ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)
        ax.plot(t, sub["DA_minus_OL"], color="#C44E52", lw=1.1)
        ax.set_title(f"{FOCUS_REGION}: {var} (DA - OL)")
        ax.set_ylabel("DA - OL")
        ax.grid(alpha=0.25)

    for ax in axes[-1, :]:
        ax.set_xlabel("Time")

    savefig_if_enabled(fig, f"snow_da_timeseries_{focus_region_tag}_landvars.png", dpi=FIG_DPI, save=SAVE_FIGURES)
    plt.show()


## Monthly Climatology of `DA-OL`

This shows the mean annual cycle of experiment differences. It is often more interpretable than the full time series for identifying seasonality in snow-DA impacts.


In [ ]:
# -------------------------
# Monthly climatology of DA-OL (regional means)
# -------------------------
# Use a small set of regions to keep the plot readable.
clim_regions = [r for r in ["Global", "NH", "NH (30-75N)"] if r in set(region_ts_df["region"])]
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

nvars = len(common_vars)
ncols = 2
nrows = int(np.ceil(nvars / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.8 * nrows), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, var in zip(axes, common_vars):
    for region_name in clim_regions:
        sub = region_ts_df[(region_ts_df["region"] == region_name) & (region_ts_df["variable"] == var)].copy()
        sub["month"] = pd.to_datetime(sub["time"]).dt.month
        clim = sub.groupby("month")["DA_minus_OL"].mean().reindex(range(1, 13))
        ax.plot(range(1, 13), clim.values, marker="o", ms=3, lw=1.2, label=region_name)

    ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)
    ax.set_title(f"DA - OL monthly climatology: {var}")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_labels, rotation=45)
    ax.grid(alpha=0.25)

for j in range(len(common_vars), len(axes)):
    axes[j].axis("off")

axes[0].legend(loc="best", fontsize=9)
fig.suptitle("M21C land-sweeper monthly climatology of DA - OL (regional tile means)", fontsize=14)
savefig_if_enabled(fig, "snow_da_monthly_climatology_regional_DA_minus_OL.png", dpi=FIG_DPI, save=SAVE_FIGURES)
plt.show()


## Seasonal `DA-OL` Maps (Lon/Lat Scatter by Tile)

These are not projected maps (no `cartopy` required), but they are quick and robust and show the main spatial structure. Each figure is one variable with DJF/MAM/JJA/SON panels.


In [ ]:
# -------------------------
# Seasonal tile maps of DA-OL
# -------------------------
seasonal_delta = {}
for var in common_vars:
    seasonal_delta[var] = maybe_compute(seasonal_mean_by_tile(d[var]))

print("Computed seasonal DA-OL means by tile for:", list(seasonal_delta.keys()))

for var in [v for v in ["SNOMASLAND", "FRLANDSNO", "TSOIL1", "SFMC", "RZMC"] if v in seasonal_delta]:
    sda = seasonal_delta[var]
    units = str(d[var].attrs.get("units", ""))
    fig = plot_lonlat_scatter_map_panels(
        sda,
        lat=lat,
        lon=lon,
        title=f"Seasonal mean impact (DA - OL): {var}",
        units=units if units else "DA - OL",
        cmap="RdBu_r",
        stride=MAP_TILE_STRIDE,
        s=4,
    )
    fname = f"seasonal_maps_DA_minus_OL_{var}.png"
    savefig_if_enabled(fig, fname, dpi=FIG_DPI, save=SAVE_FIGURES)
    plt.show()


## Spring Snow Impact vs Summer Root-Zone Soil Moisture Impact (Carryover)

This is a simple lag-style diagnostic, now split into the three configured `PERIOD_WINDOWS`.
Each row is one window.

- x-axis: yearly **MAM** mean `DA-OL` in `SNOMASLAND`
- y-axis: same-year **JJA** mean `DA-OL` in `RZMC`

Each row shows two tile-year views:
- all NH (`>=30N`) tile-years
- snow-relevant tile-years (OL MAM `FRLANDSNO > SNOW_FRAC_THRESHOLD`)


In [ ]:
# -------------------------
# Spring-to-summer carryover diagnostic (tile-year pairs), split by PERIOD_WINDOWS
# -------------------------
if not all(v in d for v in ["SNOMASLAND", "RZMC"]) or "FRLANDSNO" not in ol:
    raise RuntimeError("Need SNOMASLAND and RZMC in d, and FRLANDSNO in ol, for carryover diagnostic")

from matplotlib.ticker import MaxNLocator

# Restrict to NH extratropics where snow impacts are most likely.
carry_mask = lat >= 30.0

# Yearly seasonal means by tile (calendar-year months only; no DJF logic here).
d_snow_mam = maybe_compute(seasonal_year_mean_for_months(d["SNOMASLAND"].where(carry_mask), [3, 4, 5]))
d_rz_jja = maybe_compute(seasonal_year_mean_for_months(d["RZMC"].where(carry_mask), [6, 7, 8]))

# Use OL MAM snow fraction as a condition to isolate snow-relevant tile-year samples.
ol_fr_mam = maybe_compute(seasonal_year_mean_for_months(ol["FRLANDSNO"].where(carry_mask), [3, 4, 5]))
snow_relevant_tile_year = ol_fr_mam > SNOW_FRAC_THRESHOLD

# Align years present in all arrays.
common_years = np.intersect1d(d_snow_mam["year"].values, d_rz_jja["year"].values)
common_years = np.intersect1d(common_years, ol_fr_mam["year"].values)

d_snow_mam = d_snow_mam.sel(year=common_years)
d_rz_jja = d_rz_jja.sel(year=common_years)
snow_relevant_tile_year = snow_relevant_tile_year.sel(year=common_years)

window_bounds = build_window_bounds(PERIOD_WINDOWS)


def corr_or_nan(x, y):
    if len(x) < 3 or len(y) < 3:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def nice_symmetric_limit(arr, pct=99.0, pad_frac=0.05):
    """Robust symmetric axis limit around zero with simple rounding."""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None

    vmax = np.nanpercentile(np.abs(arr), pct)
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = np.nanmax(np.abs(arr))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    vmax = float(vmax) * (1.0 + pad_frac)

    # Nice rounding: 1, 2, 2.5, 5, 10 * 10^k
    exp10 = np.floor(np.log10(vmax))
    base = 10.0 ** exp10
    nice_mult = 10.0
    for m in (1.0, 2.0, 2.5, 5.0, 10.0):
        if m * base >= vmax:
            nice_mult = m
            break
    nice_vmax = nice_mult * base
    return (-nice_vmax, nice_vmax)


# Build sampled panel data for each window before plotting.
panel_rows = []
for wname, t0, t1 in window_bounds:
    # A carryover pair belongs to a window only if the full MAM+JJA of that year is inside the window.
    years_in_window = []
    for y in common_years.astype(int):
        mam_start = pd.Timestamp(year=int(y), month=3, day=1)
        jja_end = pd.Timestamp(year=int(y), month=8, day=31)
        if (mam_start >= t0) and (jja_end <= t1):
            years_in_window.append(int(y))

    years_in_window = np.asarray(years_in_window, dtype=int)

    if years_in_window.size == 0:
        panel_rows.append(
            {
                "window": wname,
                "start": t0,
                "end": t1,
                "years": years_in_window,
                "x_all": np.array([], dtype=float),
                "y_all": np.array([], dtype=float),
                "x_snow": np.array([], dtype=float),
                "y_snow": np.array([], dtype=float),
                "r_all": np.nan,
                "r_snow": np.nan,
            }
        )
        continue

    d_snow_w = d_snow_mam.sel(year=years_in_window)
    d_rz_w = d_rz_jja.sel(year=years_in_window)
    snow_rel_w = snow_relevant_tile_year.sel(year=years_in_window)

    x_all = d_snow_w.values.ravel()
    y_all = d_rz_w.values.ravel()

    x_snow = d_snow_w.where(snow_rel_w).values.ravel()
    y_snow = d_rz_w.where(snow_rel_w).values.ravel()

    x_all_s, y_all_s = sample_finite_pairs(x_all, y_all, max_n=SCATTER_SAMPLE_MAX, seed=RANDOM_SEED)
    x_snow_s, y_snow_s = sample_finite_pairs(x_snow, y_snow, max_n=SCATTER_SAMPLE_MAX, seed=RANDOM_SEED)

    panel_rows.append(
        {
            "window": wname,
            "start": t0,
            "end": t1,
            "years": years_in_window,
            "x_all": x_all_s,
            "y_all": y_all_s,
            "x_snow": x_snow_s,
            "y_snow": y_snow_s,
            "r_all": corr_or_nan(x_all_s, y_all_s),
            "r_snow": corr_or_nan(x_snow_s, y_snow_s),
        }
    )

# Use common axis limits across panels for easier comparison, but trim tails more aggressively
# than the original plot so the dense core is easier to read.
x_pool_parts = []
y_pool_parts = []
for pdat in panel_rows:
    if len(pdat["x_all"]) > 0:
        x_pool_parts.append(pdat["x_all"])
        y_pool_parts.append(pdat["y_all"])
    if len(pdat["x_snow"]) > 0:
        x_pool_parts.append(pdat["x_snow"])
        y_pool_parts.append(pdat["y_snow"])

x_pool = np.concatenate(x_pool_parts) if x_pool_parts else np.array([], dtype=float)
y_pool = np.concatenate(y_pool_parts) if y_pool_parts else np.array([], dtype=float)

# Tighter x percentile helps with long SNOMASLAND tails; y can stay a bit wider.
xlim = nice_symmetric_limit(x_pool, pct=97.0, pad_frac=0.03) if x_pool.size else None
ylim = nice_symmetric_limit(y_pool, pct=99.0, pad_frac=0.03) if y_pool.size else None

# Plot rows = windows, columns = (all NH>=30N, snow-relevant)
fig, axes = plt.subplots(
    len(panel_rows),
    2,
    figsize=(15, 4.4 * len(panel_rows)),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
axes = np.atleast_2d(axes)

# Column headers once (instead of repeated long titles in every panel).
col_titles = [
    "All NH>=30N tile-years",
    f"Snow-relevant tile-years (OL MAM FRLANDSNO>{SNOW_FRAC_THRESHOLD})",
]
for c, title in enumerate(col_titles):
    axes[0, c].set_title(title, fontsize=12)

for r, pdat in enumerate(panel_rows):
    row_label = f"{pdat['window']}\n{pdat['start'].date()} to {pdat['end'].date()}"

    panels = [
        (axes[r, 0], pdat["x_all"], pdat["y_all"], pdat["r_all"]),
        (axes[r, 1], pdat["x_snow"], pdat["y_snow"], pdat["r_snow"]),
    ]

    for c, (ax, x, y, rr) in enumerate(panels):
        if len(x) == 0:
            ax.text(0.5, 0.5, "No finite pairs", ha="center", va="center", transform=ax.transAxes)
            ax.grid(alpha=0.2)
            continue

        hb = ax.hexbin(x, y, gridsize=70, mincnt=1, bins="log", cmap="viridis")
        ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)
        ax.axvline(0.0, color="k", lw=0.8, alpha=0.5)
        ax.grid(alpha=0.2)

        if xlim is not None:
            ax.set_xlim(xlim)
        if ylim is not None:
            ax.set_ylim(ylim)

        # Cleaner ticks for dense multi-panel layout.
        ax.xaxis.set_major_locator(MaxNLocator(nbins=7))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=7))

        r_txt = f"{rr:.3f}" if np.isfinite(rr) else "nan"
        ax.text(
            0.02,
            0.97,
            f"n={len(x):,}\\nr={r_txt}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            color="white",
            fontsize=9,
            bbox=dict(facecolor="black", alpha=0.35, edgecolor="none"),
        )

    # Put the row/window label only once, just outside the left subplot.
    axes[r, 0].text(
        -0.16,
        0.5,
        row_label,
        transform=axes[r, 0].transAxes,
        rotation=90,
        va="center",
        ha="center",
        fontsize=10,
    )

# Shared axis labels (instead of repeating on all panels).
for ax in axes[:-1, :].ravel():
    ax.set_xlabel("")
for ax in axes[:, 1]:
    ax.set_ylabel("")

fig.supxlabel("MAM (DA - OL) SNOMASLAND [kg m$^{-2}$]")
fig.supylabel("JJA (DA - OL) RZMC [m$^3$ m$^{-3}$]")
fig.suptitle(
    "Spring snow DA impact vs summer root-zone soil moisture impact (tile-year pairs), split by window\n"
    "Shared axes use robust limits (tail-trimmed for readability)",
    fontsize=14,
)
savefig_if_enabled(fig, "carryover_MAM_dSNOMASLAND_vs_JJA_dRZMC_hexbin_by_window.png", dpi=FIG_DPI, save=SAVE_FIGURES)
plt.show()


## Optional Period Summaries (Custom Windows)

This gives compact region-mean summaries by the three windows. It is useful for checking whether snow-DA impacts changed across eras.


In [ ]:
# -------------------------
# Period summaries (regional means of DA-OL)
# -------------------------
window_bounds = build_window_bounds(PERIOD_WINDOWS)
focus_regions_for_period = [r for r in ["NH (30-75N)", "High NH (45-75N)", "Global"] if r in region_masks]

period_rows = []
for wname, t0, t1 in window_bounds:
    time_mask = (d["time"] >= np.datetime64(t0)) & (d["time"] <= np.datetime64(t1))
    d_win = d.where(time_mask, drop=True)

    for region_name in focus_regions_for_period:
        mask = region_masks[region_name]
        for var in common_vars:
            x = maybe_compute(d_win[var].where(mask).mean(dim=("time", "tile"), skipna=True))
            period_rows.append({
                "window": wname,
                "start": t0,
                "end": t1,
                "region": region_name,
                "variable": var,
                "DA_minus_OL_mean": float(x.values) if np.isfinite(x.values) else np.nan,
            })

period_df = pd.DataFrame(period_rows)
display(period_df.head(20))

# Plot a compact bar chart for the NH (30-75N).
bar_region = "NH (30-75N)"
if bar_region in set(period_df["region"]):
    plot_vars = [v for v in ["SNOMASLAND", "FRLANDSNO", "TSOIL1", "SFMC", "RZMC"] if v in set(period_df["variable"])]
    ncols = 2
    nrows = int(np.ceil(len(plot_vars) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.8 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, var in zip(axes, plot_vars):
        sub = period_df[(period_df["region"] == bar_region) & (period_df["variable"] == var)].copy()
        sub = sub.set_index("window").reindex([w[0] for w in window_bounds]).reset_index()
        x = np.arange(len(sub))
        y = sub["DA_minus_OL_mean"].to_numpy(dtype=float)
        ax.bar(x, y, color="#4C78A8", alpha=0.8)
        ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["window"], rotation=20, ha="right")
        ax.set_title(f"{bar_region}: mean DA-OL {var}")
        ax.grid(axis="y", alpha=0.25)

    for j in range(len(plot_vars), len(axes)):
        axes[j].axis("off")

    fig.suptitle("Period-mean DA-OL by window (regional tile+time mean)", fontsize=14)
    savefig_if_enabled(fig, "period_mean_DA_minus_OL_barplots.png", dpi=FIG_DPI, save=SAVE_FIGURES)
    plt.show()

# Save tables so the notebook can feed later analyses without re-computing.
p_region_ts = OUTPUT_DIR / "regional_monthly_timeseries_DA_OL_diff.csv"
p_period = OUTPUT_DIR / "period_summary_DA_minus_OL.csv"
region_ts_df.to_csv(p_region_ts, index=False)
period_df.to_csv(p_period, index=False)
print(f"Wrote table: {p_region_ts}")
print(f"Wrote table: {p_period}")


## Next Edits You May Want

- Add tile-area weighting (if you have tile areas or can derive them from the grid)
- Add region polygons (instead of latitude bands) for North America / Eurasia / basins
- Add snow-season onset/melt timing diagnostics (monthly proxy)
- Add paired comparisons to independent snow / SWE observations (if available)
